# Milestone - Unit 11: neural reranking (learning to rank the candidate pool) -- Units 2-10 each ship a retrieval *path* that sees one slice of the signal (popularity, text, behaviour, taste); Unit 6 blended their calibrated scores with **fixed** weights and Unit 10 fused a sparse + dense pair. This milestone keeps the two-stage retrieve-then-rank shape -- cheap retrieval proposes a candidate **pool**, an expensive model re-scores only that pool -- but makes the second stage **learned**: a tiny PyTorch MLP scores each pooled candidate from a 17-dim **feature vector** (the six paths' calibrated scores + content signals), trained on implicit feedback. The honest, measured story is three results: (1) the **naive training recipe LEAKS and tanks** to ~0.288 hit@10 -- below plain score-order -- because the feature-paths memorized the labels; the fix is a **time-ordered holdout inside train**. (2) The clean reranker **edges accuracy** (hit@10 ~0.364 > two-tower 0.340, ties the hybrid 0.362) but **loses coverage** (0.186 vs the Unit-6 blend's 0.333). (3) The lift is from **content features, not score-combination**: content-only is the strongest variant (0.384) and `linear ~= MLP`. Everything is fixed-seed and deterministic, and we never touch `split="test"` -- that holdout is sealed for Checkpoint B.

## 1. Load the substrate and build the candidate pool

The same generated catalog, implicit-feedback interaction log (train / val / test) and committed cold-reader partition Units 1-10 ran on. This unit **reranks the paths that already exist**, so there is no new data: the six Unit 2-10 paths are the candidate generators and the feature source. `bookrec.generated_dir()` reports a clear error if `recsys/data/generated/` has not been built.

In [ ]:
import json
from collections import defaultdict
from time import perf_counter

import bookrec
import numpy as np
import pandas as pd
from bookrec import (
    HybridRetrievalPath,
    ItemItemRetrievalPath,
    LexicalRetrievalPath,
    MatrixFactorizationPath,
    NeuralRerankerPath,
    PathRegistry,
    PopularityRetrievalPath,
    RerankerModel,
    SemanticEmbeddingRetrievalPath,
    TwoTowerRetrievalPath,
    blend,
    load_catalog,
    load_glove_subset,
    load_keywords,
    rank,
    run_blended_scoreboard,
)
from bookrec.rerank import family_mask, feature_names, split_profile_labels
from bookrec.scoreboard import _seen_and_relevant

K = 10
POOL = 50                 # pinned per-path pool size (recall ceiling ~0.772 @ pool 50)
SEED = 0
PATH_NAMES = ("popularity", "lexical", "item-item", "mf", "semantic", "two-tower")

data_dir = bookrec.generated_dir()
catalog = load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
keywords = load_keywords(data_dir / "keywords.csv.gz")
glove = load_glove_subset()

cold_readers = json.loads((data_dir / "cold_partitions.json").read_text())["cold_readers"]
interactions_path = data_dir / "interactions.csv.gz"
rows = pd.read_csv(interactions_path).to_dict("records")  # reader_id/item_id/timestamp/split/label

{
    "books": len(catalog_ids),
    "interactions": len(rows),
    "cold_readers_excluded": len(cold_readers),
    "feature_dim": len(feature_names()),
}

### Fit the six serving paths and merge one reader's pool

The reranker needs the paths **twice**: once fit on the full train (the *serving* paths used at recommendation time), and once refit on a 75% profile (the leakage-safe *training* features -- section 2). This is pass 1. For one example reader we then build the merged pool: each path's top-`POOL` calibrated candidates, unioned by book id and kept **apart per path** (absent path -> 0.0). That per-path score block is the first six columns of every candidate's feature vector.

In [ ]:
def fit_six(interaction_rows):
    """Fit the six retrieval paths whose calibrated scores feed the reranker (seed 0)."""
    return {
        "popularity": PopularityRetrievalPath().fit(interaction_rows, catalog=catalog_ids),
        "lexical": LexicalRetrievalPath(keywords).fit(interaction_rows, catalog=catalog_ids),
        "item-item": ItemItemRetrievalPath().fit(interaction_rows, catalog=catalog_ids),
        "mf": MatrixFactorizationPath(seed=SEED).fit(interaction_rows, catalog=catalog_ids),
        "semantic": SemanticEmbeddingRetrievalPath(keywords, glove).fit(interaction_rows, catalog=catalog_ids),
        "two-tower": TwoTowerRetrievalPath(seed=SEED).fit(interaction_rows, catalog=catalog_ids),
    }


# PASS 1 of 2: the SERVING paths, fit on the full train (~40 s, measured -- MF + two-tower dominate).
serving_start = perf_counter()
serving = fit_six(rows)
serving_seconds = perf_counter() - serving_start


def path_factory(train_rows):
    """The leakage-safe refit: re-fit the six paths on a given (75%-profile) row subset.

    This is a REAL refit, not a reuse of the serving paths -- training features must come from paths
    that never saw the held-out labels. Section 2 hoists this single refit out of fit() (passing
    train_paths=) only to stay in the notebook's time budget; the mechanism is exactly this callback.
    """
    return fit_six(train_rows)


reranker = NeuralRerankerPath(serving, path_factory, catalog, pool=POOL, seed=SEED)


def merge_pool(paths, reader, seen):
    """Union each path's top-POOL calibrated candidates, kept APART per path (not summed)."""
    per_item = defaultdict(dict)
    context = {"seen": seen}
    for name in PATH_NAMES:
        path = paths[name]
        for cand in path.calibrate(path.retrieve(reader, context, POOL)):
            per_item[cand.item_id][name] = cand.score
    return per_item


# One example reader's merged pool (per-path calibrated scores, kept apart -- the score feature block).
warm_readers = sorted({r["reader_id"] for r in rows if r["split"] == "train" and r["label"] == 1})
example_reader = int(warm_readers[0])
example_seen = {
    r["item_id"] for r in rows
    if r["reader_id"] == example_reader and r["split"] == "train" and r["label"] == 1
}
example_pool = merge_pool(serving, example_reader, example_seen)

print(f"serving 6-path fit: {serving_seconds:.1f}s | example reader {example_reader}: "
      f"pool of {len(example_pool)} unseen books from {len(PATH_NAMES)} paths (per-path pool={POOL})")
pd.DataFrame(
    [
        {"book": book, **{name: round(scores.get(name, 0.0), 3) for name in PATH_NAMES}}
        for book, scores in sorted(example_pool.items())[:8]
    ]
)

### The retrieval recall ceiling

The reranker only **re-orders** the pool -- if retrieval never proposes a reader's held-out relevant book, no reranker can surface it. So *any-relevant-in-pool recall* over the val cohort is a hard cap on hit@10. At the pinned `pool=50` it is ~0.772 (vs ~0.716 at pool 30 -- a tiny cost for +0.05 of ceiling, which is why we pin 50). This pass scores the val cohort only; `test` stays sealed.

In [ ]:
known_items = set(catalog_ids)
seen_by, val_relevant_by = _seen_and_relevant(interactions_path, known_items, "val")
cold_set = set(cold_readers)

ceiling_hits = []
for reader in sorted(val_relevant_by):
    seen = seen_by[reader]
    relevant = val_relevant_by[reader] - seen
    if reader in cold_set or not relevant:
        continue
    pool = merge_pool(serving, reader, seen)
    ceiling_hits.append(1.0 if any(book in relevant for book in pool) else 0.0)
pool_recall = float(np.mean(ceiling_hits))

assert pool_recall >= 0.70     # measured ~0.772 @ pool 50 -- the hard cap on ANY reranker over this pool
print(f"pool recall ceiling @pool{POOL} = {pool_recall:.3f} over {len(ceiling_hits)} val readers "
      f"-- no reranker over this pool can exceed it")

## 2. The time-ordered holdout and the 17-dim feature matrix

The headline failure mode is a training-time **leak**. If the reranker's labels are a reader's train positives *and* the feature-paths are fit on that same train, the paths have **memorized** those positives: "high score -> positive" is learned from inflated in-train scores that validation candidates never exhibit. The fix: split each reader's train positives **time-ordered** -- the latest ~25% become the reranker's **labels**, the earlier 75% is the retrieval **profile**. **Every** training-time input -- the feature-paths *and* the content/popularity statistics -- comes from that 75% profile only, so a label item never enters its own feature. This is pass 2: the paths refit on the profile rows (the held-out label items removed).

In [ ]:
profile, labels = split_profile_labels(rows, 0.25)
n_label_readers = sum(1 for reader in labels if labels[reader] and profile.get(reader))

# Remove each reader's held-out 25% label items from the training rows -> the 75% profile substrate.
profile_rows = [
    row
    for row in rows
    if not (
        row["split"] == "train"
        and row["label"] == 1
        and row["item_id"] in labels.get(row["reader_id"], set())
    )
]

# PASS 2 of 2: the single refit path_factory() describes, hoisted out for the time budget (~24 s).
profile_start = perf_counter()
profile_paths = fit_six(profile_rows)
profile_seconds = perf_counter() - profile_start

print(f"profile 6-path refit: {profile_seconds:.1f}s | {n_label_readers} readers have a "
      f"time-ordered train holdout (>=2 train positives)")
{"readers_with_label": n_label_readers, "profile_rows": len(profile_rows), "full_train_rows": len(rows)}

Now assemble the `(X, y)` training matrix **once** (~16 s -- the real cost; the MLP fit is under a second) and reuse it for every model variant below. The feature-paths are the profile refit (`train_paths=profile_paths`); the held-out 25% supplies only the positive labels, sampled against ~10 pool negatives each. Each candidate's **17-dim** vector is 6 per-path calibrated scores + 6 presence flags + `n_paths` + 4 content features (`genre_frac`, `genre_cos` vs the reader's history genre vector, `author_frac`, `log_pop`).

In [ ]:
assemble_start = perf_counter()
X, y = reranker.assemble_training_matrix(rows, train_paths=profile_paths)
assemble_seconds = perf_counter() - assemble_start

first_positive = next(i for i, label in enumerate(y) if label == 1.0)
feature_vector = pd.DataFrame(
    [{name: round(float(value), 3) for name, value in zip(feature_names(), X[first_positive])}]
)

assert X.shape[1] == 17
print(f"assemble training matrix: {assemble_seconds:.1f}s | X={X.shape} "
      f"positive-rate={y.mean():.3f} (held-out label items vs ~{reranker.n_negatives} sampled "
      f"pool negatives each)")
feature_vector

## 3. The leakage trap: naive tanks, clean works

Train the reranker two ways over the **same** pool and score both on val. The **clean** recipe (labels = held-out 25%, every feature from the 75% profile) lands at ~0.364. The **leaky** recipe (the anti-pattern: feature-paths are the full-fit serving paths that memorized the label items, counts from the full train) **tanks to ~0.288** -- below even the fixed 6-way score-order over the pool it reranks. That is the measured cost of letting the features see the labels.

In [ ]:
# CLEAN recipe: the MLP over all 17 features (fit < 1 s -- assembly, not the fit, is the cost).
mlp_fit_start = perf_counter()
clean_model = RerankerModel(seed=SEED).fit(X, y)
mlp_fit_seconds = perf_counter() - mlp_fit_start
reranker.set_model(clean_model)


def board(path, pool=K):
    """hit@10 + catalog coverage on the frozen val split (single-path blended scoreboard)."""
    result = run_blended_scoreboard(
        {path.name: path}, interactions_path, catalog_ids=catalog_ids, k=K, pool=pool,
        cold_readers=cold_readers,
    )
    return result.hit_rate_at_k, result.catalog_coverage


clean_hit, clean_cov = board(reranker)

# The fixed 6-way equal-weight score-order over the SAME pool (the baseline the learned ranker beats).
six_way = run_blended_scoreboard(
    {name: serving[name] for name in PATH_NAMES}, interactions_path, catalog_ids=catalog_ids,
    k=K, pool=POOL, cold_readers=cold_readers,
)
score_order_hit = six_way.hit_rate_at_k

# LEAKY recipe (anti-pattern): train on the full-fit serving paths + full-train counts.
leaky = NeuralRerankerPath(serving, path_factory, catalog, pool=POOL, seed=SEED, leaky=True)
leaky.fit(rows)
leaky_hit, leaky_cov = board(leaky)

print(f"MLP fit: {mlp_fit_seconds:.2f}s")
assert leaky_hit < clean_hit          # the leak tanks the reranker below the clean recipe ...
assert leaky_hit < score_order_hit    # ... and below the fixed score-order it is supposed to beat
pd.DataFrame([
    {"recipe": "leaky (naive: labels = train positives, paths fit on same train)",
     "hit@10": round(leaky_hit, 4), "coverage": round(leaky_cov, 4)},
    {"recipe": "fixed 6-way score-order over the pool",
     "hit@10": round(score_order_hit, 4), "coverage": round(six_way.catalog_coverage, 4)},
    {"recipe": "clean (time-ordered holdout; features from 75% profile)",
     "hit@10": round(clean_hit, 4), "coverage": round(clean_cov, 4)},
])

## 4. The val scoreboard: accuracy up, coverage down

Register the clean reranker as `reranker-v1` and read it honestly against the best single path (two-tower), the fixed score-order of the pool, the broad Unit-6 blend and the Unit-10 hybrid -- on **both** hit@10 and catalog coverage. The reranker edges the two-tower (0.364 > 0.340) and ties the hybrid (0.362), clearing the gate (`>= two_tower - 0.01` and `>= score_order`). But its coverage (0.186) is a real **loss** against the Unit-6 blend's 0.333: a precise ranker concentrates its picks. Neither recommender dominates -- the reranker is sharper, the blend is broader.

In [ ]:
registry = PathRegistry()
registry.register(reranker)
assert reranker.artifact_name() == "reranker-v1"

two_tower_hit, two_tower_cov = board(serving["two-tower"])
hybrid = HybridRetrievalPath(serving["lexical"], serving["two-tower"], weight=0.7, pool=50).fit(rows)
hybrid_hit, hybrid_cov = board(hybrid)
u6 = run_blended_scoreboard(
    {name: serving[name] for name in ("popularity", "lexical", "item-item", "mf")},
    interactions_path, catalog_ids=catalog_ids, k=K, pool=30,
    weights={"popularity": 0.25, "lexical": 0.5, "item-item": 1.0, "mf": 1.0},
    cold_readers=cold_readers,
)

# GATE: the clean reranker is not tanked vs the two-tower AND beats the fixed score-order pool.
assert clean_hit >= two_tower_hit - 0.01
assert clean_hit >= score_order_hit
# Accuracy ties/edges the hybrid (directional, within noise) ...
assert clean_hit >= hybrid_hit - 0.01
# ... but coverage is an honest LOSS vs the broad Unit-6 blend.
assert clean_cov < u6.catalog_coverage

pd.DataFrame([
    {"recommender": "two-tower (U8, best single path)", "hit@10": round(two_tower_hit, 4), "coverage": round(two_tower_cov, 4)},
    {"recommender": "6-way score-order over the pool", "hit@10": round(score_order_hit, 4), "coverage": round(six_way.catalog_coverage, 4)},
    {"recommender": "U6 blend (4-way, broad)", "hit@10": round(u6.hit_rate_at_k, 4), "coverage": round(u6.catalog_coverage, 4)},
    {"recommender": "U10 hybrid (sparse+dense)", "hit@10": round(hybrid_hit, 4), "coverage": round(hybrid_cov, 4)},
    {"recommender": "reranker (learned, clean recipe)", "hit@10": round(clean_hit, 4), "coverage": round(clean_cov, 4)},
])

## 5. What carries the lift: content features, and `linear ~= MLP`

The counterintuitive payoff. Swap in two ablated models on the same assembled matrix: a **content-only** reranker (just the 4 content features) and a **scores-only** one (the path scores + presence + `n_paths`). Content-only is the **strongest** variant (~0.384); dropping the path scores does not hurt, and scores-only falls back toward plain score-order. And a plain **linear** logistic combiner ties the MLP (~0.366 vs 0.364) -- the non-linearity adds nothing here. The lesson: a learned ranker wins by seeing *features a single path can't carry across readers* (content affinity to the reader's history), not by cleverly combining correlated scores.

In [ ]:
def board_model(model):
    """Score val with a model installed on the clean reranker's serving machinery, then restore."""
    original = reranker.model
    try:
        reranker.set_model(model)
        return board(reranker)
    finally:
        reranker.set_model(original)


content_model = RerankerModel(seed=SEED).fit(X, y, feature_mask=family_mask(("content",)))
scores_model = RerankerModel(seed=SEED).fit(X, y, feature_mask=family_mask(("scores", "presence", "meta")))
linear_model = RerankerModel(seed=SEED, linear=True).fit(X, y)

content_hit, content_cov = board_model(content_model)
scores_hit, scores_cov = board_model(scores_model)
linear_hit, linear_cov = board_model(linear_model)

assert content_hit >= scores_hit              # content carries the lift; path scores barely matter
assert abs(linear_hit - clean_hit) < 0.05     # linear ~= MLP: the non-linear combiner adds nothing
pd.DataFrame([
    {"reranker variant": "content-only (4 content features)", "hit@10": round(content_hit, 4), "coverage": round(content_cov, 4)},
    {"reranker variant": "scores-only (paths + presence + n_paths)", "hit@10": round(scores_hit, 4), "coverage": round(scores_cov, 4)},
    {"reranker variant": "linear logistic (all features)", "hit@10": round(linear_hit, 4), "coverage": round(linear_cov, 4)},
    {"reranker variant": "MLP (all features)", "hit@10": round(clean_hit, 4), "coverage": round(clean_cov, 4)},
])

### One reader the reranker visibly improves

Concretely: the first val reader (in id order) for whom the learned reranker pulls a held-out relevant book into the top-10 that the fixed 6-way score-order over the **same pool** ranked out. Same candidates, different ordering -- the learned content affinity is what promotes the hit.

In [ ]:
improved = None
for reader in sorted(val_relevant_by):
    seen = seen_by[reader]
    relevant = val_relevant_by[reader] - seen
    if reader in cold_set or not relevant:
        continue
    per_path = {name: serving[name].retrieve(reader, {"seen": seen}, POOL) for name in PATH_NAMES}
    order_recs = [c.item_id for c in rank(blend(per_path), n=K, exclude=seen)]
    rerank_recs = [c.item_id for c in reranker.retrieve(reader, {"seen": seen}, K)]
    if (set(rerank_recs) & relevant) and not (set(order_recs) & relevant):
        improved = (reader, relevant, order_recs, rerank_recs)
        break

assert improved is not None
reader, relevant, order_recs, rerank_recs = improved
hit_book = sorted(set(rerank_recs) & relevant)[0]
hit_rank = next(j for j, book in enumerate(rerank_recs, 1) if book == hit_book)
print(f"reader {reader}: the reranker surfaces held-out book {hit_book} at rank "
      f"{hit_rank} of 10 -- the 6-way score-order missed it from the same pool.")
pd.DataFrame({
    "score-order top-10": [f"{b}{'  <-- relevant' if b in relevant else ''}" for b in order_recs],
    "reranker top-10": [f"{b}{'  <-- relevant' if b in relevant else ''}" for b in rerank_recs],
})

**The honest headline.** A learned second-stage **reranker** over the merged candidate pool edges the best single path on hit@10 (0.364 > two-tower 0.340, ties the Unit-10 hybrid 0.362) and clears its gate against the fixed score-order it reranks -- but at a real **coverage loss** (0.186 vs the Unit-6 blend's 0.333). The win comes from **content features** (content-only 0.384 is the best variant), not from combining the correlated path scores, and `linear ~= MLP`. The headline lesson is the **leakage trap**: the naive recipe (labels = train positives, feature-paths fit on that same train) memorizes the labels and **tanks to 0.288**; the **time-ordered holdout inside train** -- every training-time feature built from the 75% profile, the held-out 25% supplying only labels -- is what makes it work. And the reranker only re-orders the pool, so the retrieval **recall ceiling** (~0.772 @ pool 50) caps it.

**The bridge.** Unit 12 adds **sequence** features (the order of a reader's history); Unit 13 revisits the fairness and coverage cost of a learned ranker; Unit 14's capstone wires retrieval -> blend -> rerank end to end.